# Notebook 0: Environment Setup & Audio Fundamentals

**What:** Verify the Demucs environment and understand core audio concepts (waveform, spectrogram, mix = sum of sources).

**Why:** Source separation models operate on audio representations. Understanding STFT and the supervised setup is essential.

**How:** Run checks, load a simple audio, visualize time vs frequency domain.

## 1. Environment Check

Verify PyTorch, CUDA, and Demucs are installed. For 3070 Ti: expect CUDA 11.x/12.x and ~8GB VRAM.

In [ ]:
import torch
import torchaudio
import sys

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Add demucs to path if running from repo
sys.path.insert(0, r'D:\demucs')
try:
    import demucs
    print(f"Demucs: OK")
except ImportError:
    print("Demucs: Run 'pip install demucs' or 'pip install -e .' from repo root")

## 2. Waveform vs Spectrogram — First Principles

- **Waveform:** Raw audio samples `x[t]`. Shape: `(channels, samples)`.
- **Spectrogram:** Short-Time Fourier Transform (STFT). Represents frequency content over time. Shape: `(freq_bins, time_frames)`.

**Key relation:** `mix = drums + bass + vocals + other` (supervised learning setup)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create synthetic "mix" from two sources (conceptual)
sr = 44100
duration = 1.0  # seconds
t = np.linspace(0, duration, int(sr * duration), dtype=np.float32)

source1 = 0.5 * np.sin(2 * np.pi * 440 * t)   # A4
source2 = 0.3 * np.sin(2 * np.pi * 880 * t)  # A5
mix = source1 + source2  # Linear mix

print("Waveform shapes:", source1.shape, mix.shape)
print("Mix = source1 + source2 (by definition in MSS)")

In [ ]:
# STFT using PyTorch (same idea as demucs.spec.spectro)
x = torch.from_numpy(mix).unsqueeze(0).unsqueeze(0)  # (1, 1, T)
n_fft = 1024
hop = 256
spec = torch.stft(x.squeeze(0), n_fft, hop_length=hop, return_complex=True)
mag = torch.abs(spec)

print("Spectrogram shape (freq_bins, time_frames):", mag.shape)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
axes[0].plot(t[:2000], mix[:2000])
axes[0].set_title("Waveform (time domain)")
axes[0].set_xlabel("Time (s)")

axes[1].imshow(mag.squeeze().numpy(), aspect='auto', origin='lower', cmap='magma')
axes[1].set_title("Spectrogram (time-frequency)")
axes[1].set_xlabel("Time frame")
axes[1].set_ylabel("Frequency bin")
plt.tight_layout()
plt.show()

## 3. Demucs STFT Wrapper

Demucs uses `demucs.spec.spectro` and `ispectro` for hybrid models. Check that we can call them.

In [ ]:
from demucs.spec import spectro, ispectro

x = torch.randn(2, 1, 44100)  # (batch, channels, samples)
z = spectro(x, n_fft=1024)
x_recon = ispectro(z, hop_length=256, length=x.shape[-1])

print("Input shape:", x.shape)
print("Spectrogram shape:", z.shape)
print("Reconstructed shape:", x_recon.shape)
print("Reconstruction error (approx):", (x - x_recon).abs().mean().item())

## 4. Summary

- **Waveform:** Raw samples; Demucs time branch works here.
- **Spectrogram:** STFT; Demucs frequency branch + hybrid models use it.
- **Mix:** Linear sum of sources. Model learns to invert this.

**Next:** Notebook 1 — Audio I/O and preprocessing in Demucs.